In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import re

# --- CONFIGURATION ---
root_output_dir = 'final_temporal_analysis'
os.makedirs(root_output_dir, exist_ok=True)

# Helper function to sanitize folder names (remove special characters)
def sanitize_filename(name):
    return re.sub(r'[<>:"/\\|?*]', '_', str(name)).strip()

# --- 1. DATA LOADING & PREPROCESSING ---
print("📥 Loading data...")
try:
    df = pd.read_csv('./filtered_envisoft_air_quality_weather_data.csv', parse_dates=['Timestamp'], dayfirst=True)
except FileNotFoundError:
    print("❌ Error: File 'filtered_envisoft_air_quality_weather_data.csv' not found.")
    exit()

# Convert AQI to numeric and handle missing values
df['AQI'] = pd.to_numeric(df['AQI'], errors='coerce')
df = df.dropna(subset=['AQI', 'Name'])  # Ensure we have AQI and Station Name

# Extract time features
df['Hour'] = df['Timestamp'].dt.hour
df['Month_Period'] = df['Timestamp'].dt.to_period('M')

# Get unique stations
stations = df['Name'].unique()
print(f"👉 Found {len(stations)} stations. Starting processing...\n")

# --- 2. PROCESSING LOOP ---
for station_name in stations:
    # Safe folder name
    clean_station_name = sanitize_filename(station_name)
    print(f"📊 Processing: {clean_station_name}")
    
    # Filter data for this station
    station_data = df[df['Name'] == station_name].copy()
    
    # Create Station Folder
    station_dir = os.path.join(root_output_dir, clean_station_name)
    os.makedirs(station_dir, exist_ok=True)

    # ==========================================
    # PART A: Full Time Series (All Data)
    # ==========================================
    plt.figure(figsize=(15, 6))
    plt.plot(station_data['Timestamp'], station_data['AQI'], label='AQI', color='teal', linewidth=1)
    
    plt.title(f"Full Time Series: AQI History\n{station_name}")
    plt.xlabel("Date")
    plt.ylabel("AQI")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    
    # Save
    plt.savefig(os.path.join(station_dir, '1_time_series_full.png'))
    plt.close()

    # ==========================================
    # PART B: Overall Hourly Average (Aggregated)
    # ==========================================
    # Group by Hour (0-23) across ALL dates
    hourly_overall = station_data.groupby('Hour')['AQI'].agg(['mean', 'std']).reset_index()
    
    plt.figure(figsize=(12, 6))
    plt.plot(hourly_overall['Hour'], hourly_overall['mean'], label='Mean AQI', color='darkblue', linewidth=2)
    
    # Fill Standard Deviation (Confidence Interval approximation)
    if hourly_overall['std'].notna().any():
        upper = hourly_overall['mean'] + hourly_overall['std']
        lower = hourly_overall['mean'] - hourly_overall['std']
        # Clip lower bound to 0 if AQI cannot be negative
        lower = lower.clip(lower=0)
        plt.fill_between(hourly_overall['Hour'], lower, upper, color='darkblue', alpha=0.15, label='Standard Deviation')

    plt.title(f"Overall Hourly Average (Diurnal Cycle)\n{station_name}")
    plt.xlabel("Hour of Day (0-23)")
    plt.ylabel("Average AQI")
    plt.xticks(range(0, 24))
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    
    # Save
    plt.savefig(os.path.join(station_dir, '2_hourly_average_overall.png'))
    plt.close()

    # ==========================================
    # PART C: Monthly Hourly Average (Per Month)
    # ==========================================
    # Create a subfolder for monthly plots to keep it tidy (optional, but good for many months)
    # or just save them in the main station folder as requested.
    
    unique_months = station_data['Month_Period'].unique()
    
    for month in unique_months:
        month_data = station_data[station_data['Month_Period'] == month]
        
        # Group by Hour for this specific month
        hourly_month = month_data.groupby('Hour')['AQI'].agg(['mean', 'std']).reset_index()
        
        plt.figure(figsize=(12, 6))
        plt.plot(hourly_month['Hour'], hourly_month['mean'], label=f'Mean AQI ({month})', color='firebrick', linewidth=2)
        
        # Std Dev
        if hourly_month['std'].notna().any():
            upper = hourly_month['mean'] + hourly_month['std']
            lower = hourly_month['mean'] - hourly_month['std']
            lower = lower.clip(lower=0)
            plt.fill_between(hourly_month['Hour'], lower, upper, color='firebrick', alpha=0.15, label='Standard Deviation')

        plt.title(f"Hourly Average: {month}\n{station_name}")
        plt.xlabel("Hour of Day (0-23)")
        plt.ylabel("Average AQI")
        plt.xticks(range(0, 24))
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend()
        plt.tight_layout()
        
        # Save: e.g., 3_hourly_average_2025-04.png
        filename = f'3_hourly_average_{month}.png'
        plt.savefig(os.path.join(station_dir, filename))
        plt.close()

print("-" * 50)
print(f"✅ Processing complete! Check the folder: '{root_output_dir}'")

📥 Loading data...
👉 Found 13 stations. Starting processing...

📊 Processing: Hà Nội_ 556 Nguyễn Văn Cừ (KK)
📊 Processing: HCM_ Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận 2 (Ngã ba Lê Hữu Kiểu và Trương Văn Bang) (KK)
📊 Processing: Long An_ UBND Tp Tân An - 76 Hùng Vương - P.2 (KK)
📊 Processing: Hà Nội_ ĐHBK cổng Parabol đường Giải Phóng (KK)
📊 Processing: Hà Nội_ Công viên Nhân Chính - Khuất Duy Tiến (KK)
📊 Processing: Đà Nẵng_ Khuôn viên trường ĐH sư phạm Đà Nẵng (KK)
📊 Processing: Thái nguyên_ Đường Hùng Vương - Tp Thái Nguyên (KK)
📊 Processing: Phú Thọ_ đường Hùng Vương - Tp Việt Trì (KK)
📊 Processing: Bắc Giang_ Khu liên cơ quan tỉnh Bắc Giang - P. Ngô Quyền - TP. Bắc Giang (KK)
📊 Processing: Hà Nam_ Công Viên Nam Cao - P.Quang Trung - TP. Phủ Lý (KK)
📊 Processing: Bình Dương_ số 593 Đại lộ Bình Dương, P. Hiệp Thành (KK)
📊 Processing: Quảng Bình_ Khu kinh tế Hòn La (KK)
📊 Processing: HCM_ Khu Liên cơ quan Bộ Tài Nguyên và Môi Trường - số 20 Đ. Lý Chính Thắng (KK)
-------------------

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import math
import os

# --- CONFIGURATION ---
INPUT_FILE = './filtered_envisoft_air_quality_weather_data.csv'
OUTPUT_DIR = 'final_temporal_by_region_analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- REGION MAPPING LOGIC ---
# Keywords to look for in Station Names to assign regions
REGION_KEYWORDS = {
    'North': ['Hà Nội', 'Bắc Giang', 'Thái Nguyên', 'Phú Thọ', 'Hà Nam', 'Bắc Ninh', 'Hải Dương', 'Quảng Ninh'],
    'Central': ['Đà Nẵng', 'Huế', 'Quảng Bình', 'Thanh Hóa', 'Nghệ An', 'Hà Tĩnh'],
    'South': ['HCM', 'Hồ Chí Minh', 'Bình Dương', 'Long An', 'Đồng Nai', 'Vũng Tàu', 'Cần Thơ']
}

def get_region(name):
    """Returns 'North', 'Central', or 'South' based on name keywords."""
    for region, keywords in REGION_KEYWORDS.items():
        for kw in keywords:
            if kw.lower() in name.lower():
                return region
    return 'Other' # For stations that don't match (fallback)

# --- 1. DATA LOADING ---
print("📥 Loading data...")
try:
    df = pd.read_csv(INPUT_FILE, parse_dates=['Timestamp'], dayfirst=True)
except FileNotFoundError:
    print(f"❌ Error: File '{INPUT_FILE}' not found.")
    exit()

df['AQI'] = pd.to_numeric(df['AQI'], errors='coerce')
df = df.dropna(subset=['AQI', 'Name'])

# Assign Regions
df['Region'] = df['Name'].apply(get_region)

# Sort by Region then Name to keep things tidy
df = df.sort_values(by=['Region', 'Name'])

print(f"👉 Classified stations into regions. Generating {len(df['Region'].unique())} dashboards...\n")

# --- 2. GENERATE DASHBOARDS BY REGION ---
for region in ['North', 'Central', 'South']:
    
    # Filter for this region
    region_data = df[df['Region'] == region]
    
    if region_data.empty:
        print(f"⚠️ No stations found for {region}. Skipping.")
        continue
        
    stations = region_data['Name'].unique()
    n_stations = len(stations)
    
    print(f"📊 Generating {region} Dashboard ({n_stations} stations)...")

    # Layout Calculation (Grid)
    ncols = 3
    nrows = math.ceil(n_stations / ncols)
    
    # Dynamic Figure Height (bigger if more stations)
    fig_height = 4 * nrows
    if fig_height < 6: fig_height = 6 # Minimum height
    
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(18, fig_height), sharex=True, sharey=True)
    
    # Handle single row case (axes is 1D) or single plot (axes is object)
    if n_stations == 1:
        axes = [axes]
    elif nrows == 1 or ncols == 1:
        axes = axes.flatten()
    else:
        axes = axes.flatten()

    # --- PLOT EACH STATION ---
    for i, station in enumerate(stations):
        ax = axes[i]
        
        # Get data for this station
        st_data = region_data[region_data['Name'] == station].sort_values('Timestamp')
        
        # Plot
        ax.plot(st_data['Timestamp'], st_data['AQI'], 
                color='#d62728' if region == 'North' else '#1f77b4' if region == 'South' else '#ff7f0e', 
                linewidth=1.2)
        
        ax.fill_between(st_data['Timestamp'], st_data['AQI'], 0, alpha=0.1, 
                        color='#d62728' if region == 'North' else '#1f77b4' if region == 'South' else '#ff7f0e')

        # Formatting
        ax.set_title(station, fontsize=10, fontweight='bold')
        ax.grid(True, linestyle='--', alpha=0.5)
        
        # Threshold Lines
        ax.axhline(150, color='red', linestyle='--', linewidth=0.8, alpha=0.5) # Unhealthy
        ax.axhline(50, color='green', linestyle='--', linewidth=0.8, alpha=0.5) # Good

    # Hide empty subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle(f"Regional Air Quality Analysis: {region} Vietnam\n(AQI Time Series)", fontsize=18, y=1.02)
    plt.tight_layout()
    
    # Save
    save_path = os.path.join(OUTPUT_DIR, f'Dashboard_AQI_{region}.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

print("-" * 50)
print(f"✅ Regional Dashboards saved to: '{OUTPUT_DIR}'")

📥 Loading data...
👉 Classified stations into regions. Generating 3 dashboards...

📊 Generating North Dashboard (7 stations)...
📊 Generating Central Dashboard (2 stations)...
📊 Generating South Dashboard (4 stations)...
--------------------------------------------------
✅ Regional Dashboards saved to: 'final_temporal_analysis'


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- CONFIGURATION ---
INPUT_FILE = './filtered_envisoft_air_quality_weather_data.csv'
OUTPUT_DIR = 'final_monthly_hourly_comparison'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 1. DATA LOADING ---
print("📥 Loading data...")
try:
    df = pd.read_csv(INPUT_FILE, parse_dates=['Timestamp'], dayfirst=True)
except FileNotFoundError:
    print(f"❌ Error: File '{INPUT_FILE}' not found.")
    exit()

df['AQI'] = pd.to_numeric(df['AQI'], errors='coerce')
df = df.dropna(subset=['AQI', 'Name'])

# Extract Time Features
df['Month_Period'] = df['Timestamp'].dt.to_period('M') # e.g., "2025-05"
df['Hour'] = df['Timestamp'].dt.hour

# --- 2. DEFINE REGIONS ---
def get_region(name):
    name_lower = str(name).lower()
    if any(x in name_lower for x in ['hà nội', 'bắc', 'thái nguyên', 'phú thọ', 'hải dương']):
        return 'North (Hanoi Region)'
    elif any(x in name_lower for x in ['hcm', 'hồ chí minh', 'bình dương', 'đông nam bộ', 'long an']):
        return 'South (HCM Region)'
    return None

df['Region'] = df['Name'].apply(get_region)
df = df.dropna(subset=['Region']) # Keep only classified data

# --- 3. GENERATE 1 FILE PER MONTH ---
unique_months = sorted(df['Month_Period'].unique())
print(f"👉 Found {len(unique_months)} months. Generating comparison plots...\n")

for month in unique_months:
    month_str = str(month)
    print(f"📊 Processing: {month_str}...")
    
    # Filter for this month
    month_data = df[df['Month_Period'] == month]
    
    # Setup Figure (2 Subplots: Left=North, Right=South)
    fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)
    
    # --- PLOT NORTH (Hanoi) ---
    north_data = month_data[month_data['Region'] == 'North (Hanoi Region)']
    ax_north = axes[0]
    
    if not north_data.empty:
        # Plot each station as a faint line
        sns.lineplot(data=north_data, x='Hour', y='AQI', hue='Name', 
                     ax=ax_north, legend=False, alpha=0.3, linewidth=1, palette='Reds')
        # Plot the REGIONAL AVERAGE as a thick line
        sns.lineplot(data=north_data, x='Hour', y='AQI', 
                     ax=ax_north, color='darkred', linewidth=3, label='Regional Mean')
        
        ax_north.set_title(f"NORTH (Hanoi): {month_str}\n(Diurnal Cycle)", fontsize=14, fontweight='bold', color='darkred')
    else:
        ax_north.text(0.5, 0.5, "No Data", ha='center', fontsize=15)

    # --- PLOT SOUTH (HCM) ---
    south_data = month_data[month_data['Region'] == 'South (HCM Region)']
    ax_south = axes[1]
    
    if not south_data.empty:
        # Plot each station as a faint line
        sns.lineplot(data=south_data, x='Hour', y='AQI', hue='Name', 
                     ax=ax_south, legend=False, alpha=0.3, linewidth=1, palette='Blues')
        # Plot the REGIONAL AVERAGE as a thick line
        sns.lineplot(data=south_data, x='Hour', y='AQI', 
                     ax=ax_south, color='darkblue', linewidth=3, label='Regional Mean')
        
        ax_south.set_title(f"SOUTH (HCM): {month_str}\n(Diurnal Cycle)", fontsize=14, fontweight='bold', color='darkblue')
    else:
        ax_south.text(0.5, 0.5, "No Data", ha='center', fontsize=15)

    # --- FORMATTING ---
    for ax in axes:
        ax.set_xlabel("Hour of Day (0-23)")
        ax.set_ylabel("AQI")
        ax.set_xticks(range(0, 24))
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.set_ylim(0, 250) # Fixed scale for easy comparison across months
        
        # Highlight "Rush Hours" (7-9 AM & 5-7 PM)
        ax.axvspan(7, 9, color='gray', alpha=0.1)
        ax.axvspan(17, 19, color='gray', alpha=0.1)

    plt.tight_layout()
    
    # Save
    save_path = os.path.join(OUTPUT_DIR, f'Hourly_Comparison_{month_str}.png')
    plt.savefig(save_path, dpi=150)
    plt.close()

print("-" * 50)
print(f"✅ Images saved to: '{OUTPUT_DIR}'")

📥 Loading data...
👉 Found 9 months. Generating comparison plots...

📊 Processing: 2025-04...
📊 Processing: 2025-05...
📊 Processing: 2025-06...
📊 Processing: 2025-07...
📊 Processing: 2025-08...
📊 Processing: 2025-09...
📊 Processing: 2025-10...
📊 Processing: 2025-11...
📊 Processing: 2025-12...
--------------------------------------------------
✅ Images saved to: 'final_monthly_hourly_comparison'


In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import math

# --- CONFIGURATION ---
INPUT_FILE = './filtered_envisoft_air_quality_weather_data.csv'
BASE_OUTPUT_DIR = 'final_station_verification_with_std'
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

# --- 1. DATA LOADING ---
print("📥 Loading data...")
try:
    df = pd.read_csv(INPUT_FILE, parse_dates=['Timestamp'], dayfirst=True)
except FileNotFoundError:
    print(f"❌ Error: File '{INPUT_FILE}' not found.")
    exit()

df['AQI'] = pd.to_numeric(df['AQI'], errors='coerce')
df = df.dropna(subset=['AQI', 'Name'])

# Extract Time Features
df['Month_Period'] = df['Timestamp'].dt.to_period('M')
df['Hour'] = df['Timestamp'].dt.hour

# --- 2. DEFINE REGIONS ---
def get_city_group(name):
    name_lower = str(name).lower()
    # Expanded slightly to ensure we capture all relevant stations
    if any(x in name_lower for x in ['hà nội']):
        return 'Hanoi_Region'
    elif any(x in name_lower for x in ['hcm', 'hồ chí minh']):
        return 'HCM_Region'
    return None

df['City_Group'] = df['Name'].apply(get_city_group)
df = df.dropna(subset=['City_Group']) # Keep only classified data

# --- 3. GENERATE DASHBOARDS ---
unique_months = sorted(df['Month_Period'].unique())
groups = ['Hanoi_Region', 'HCM_Region']

print(f"👉 Found {len(unique_months)} months. Generating station dashboards...\n")

for group in groups:
    # Create Folder
    group_dir = os.path.join(BASE_OUTPUT_DIR, group)
    os.makedirs(group_dir, exist_ok=True)
    
    # Filter data for this city group
    group_data = df[df['City_Group'] == group]
    stations = sorted(group_data['Name'].unique())
    n_stations = len(stations)
    
    if n_stations == 0:
        continue

    print(f"🏙️ Processing {group} ({n_stations} stations)...")

    # Loop through each month
    for month in unique_months:
        month_str = str(month)
        month_data = group_data[group_data['Month_Period'] == month]
        
        if month_data.empty:
            continue

        # --- CALCULATE LAYOUT ---
        ncols = 3
        nrows = math.ceil(n_stations / ncols)
        fig_height = max(5, 3 * nrows) 
        
        fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(18, fig_height), sharex=True, sharey=True)
        axes = axes.flatten() if n_stations > 1 else [axes]

        # --- PLOT EACH STATION ---
        for i, station in enumerate(stations):
            ax = axes[i]
            
            # Get specific station data for this month
            st_data = month_data[month_data['Name'] == station]
            
            if not st_data.empty:
                # Group by Hour to get Mean AND Standard Deviation
                hourly_stats = st_data.groupby('Hour')['AQI'].agg(['mean', 'std'])
                
                # Define Color
                color = '#d62728' if group == 'Hanoi_Region' else '#1f77b4' # Red for North, Blue for South
                
                # 1. Plot the MEAN Line (Thick line)
                ax.plot(hourly_stats.index, hourly_stats['mean'], marker='.', markersize=3, 
                        color=color, linewidth=2, label='Mean AQI')
                
                # 2. Plot the STANDARD DEVIATION (Shaded Area)
                # Fill NaN std with 0 (happens if only 1 data point exists for that hour)
                std = hourly_stats['std'].fillna(0)
                upper_bound = hourly_stats['mean'] + std
                lower_bound = (hourly_stats['mean'] - std).clip(lower=0) # AQI cannot be negative
                
                ax.fill_between(hourly_stats.index, lower_bound, upper_bound, 
                                color=color, alpha=0.2, label='Std Dev (Variance)')
                
                # Aesthetics
                ax.set_title(station, fontsize=9, fontweight='bold')
                ax.grid(True, linestyle=':', alpha=0.6)
                ax.set_xticks([0, 6, 12, 18, 23])
                
                # Highlight Rush Hours
                ax.axvspan(7, 9, color='gray', alpha=0.1) 
                ax.axvspan(17, 19, color='gray', alpha=0.1)
            else:
                ax.text(0.5, 0.5, "No Data", ha='center', color='gray')
                ax.set_title(station, fontsize=9, color='gray')

        # Hide empty subplots
        for j in range(i + 1, len(axes)):
            fig.delaxes(axes[j])

        plt.suptitle(f"{group}: Station Comparison - {month_str}\n(Hourly Mean ± Standard Deviation)", fontsize=16, y=1.02)
        plt.tight_layout()
        
        # Save
        filename = f'Dashboard_{month_str}.png'
        save_path = os.path.join(group_dir, filename)
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
        plt.close()

print("-" * 50)
print(f"✅ Dashboards with Std Dev generated in: '{BASE_OUTPUT_DIR}'")

📥 Loading data...
👉 Found 9 months. Generating station dashboards...

🏙️ Processing Hanoi_Region (3 stations)...
🏙️ Processing HCM_Region (2 stations)...
--------------------------------------------------
✅ Dashboards with Std Dev generated in: 'final_station_verification_with_std'


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import os

# --- 1. SETUP ---
# Path for saving the output maps
OUTPUT_DIR = 'output/spatial_visualization'
# Cleaned data from the normalization script
INPUT_FILE = './filtered_envisoft_air_quality_data.csv'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. LOAD AND CLEAN DATA ---
print("📥 Loading data...")
# Read file and ensure Timestamp is recognized as a date
df = pd.read_csv(INPUT_FILE, parse_dates=['Timestamp'], dayfirst=True)

# Make sure AQI and coordinates are treated as numbers
df['AQI'] = pd.to_numeric(df['AQI'], errors='coerce')
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Remove any rows missing these essential values
df = df.dropna(subset=['AQI', 'Latitude', 'Longitude', 'Name'])

# --- 3. CALCULATE AVERAGES ---
# Get the average AQI for each station based on its location
station_avg_aqi = df.groupby('Name').agg({
    'AQI': 'mean',
    'Latitude': 'first',
    'Longitude': 'first'
}).reset_index()

# --- 4. STATIC MAP (Matplotlib) ---
print("🎨 Creating static map...")
plt.figure(figsize=(12, 8))

# Plot points colored by AQI (Green = Good, Red = Bad)
scatter = plt.scatter(
    station_avg_aqi['Longitude'], station_avg_aqi['Latitude'],
    c=station_avg_aqi['AQI'], cmap='RdYlGn_r', s=150, edgecolors='black', alpha=0.7
)

# Add short name labels to each station point
for i, row in station_avg_aqi.iterrows():
    plt.text(row['Longitude'], row['Latitude'], row['Name'].split(':')[0], fontsize=8, ha='right')

plt.colorbar(scatter, label='Average AQI')
plt.title('Air Quality Stations: Average AQI Distribution')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.savefig(os.path.join(OUTPUT_DIR, 'aqi_spatial_distribution.png'), dpi=150)
print(f"✅ Saved: {OUTPUT_DIR}/aqi_spatial_distribution.png")

# --- 5. INTERACTIVE MAP (Plotly) ---
print("🌐 Creating interactive map...")
fig = px.scatter_geo(
    station_avg_aqi,
    lat='Latitude',
    lon='Longitude',
    color='AQI',
    size='AQI',
    hover_name='Name',
    color_continuous_scale='RdYlGn_r',
    title='Interactive Station Map: Average AQI'
)

# Focus the map view on Vietnam
fig.update_layout(
    geo=dict(
        center=dict(lat=15, lon=107),
        projection_scale=5,
        scope='asia',
        showland=True,
        landcolor='LightGray'
    )
)

fig.write_html(os.path.join(OUTPUT_DIR, 'aqi_spatial_distribution_interactive.html'))
print(f"✅ Saved: {OUTPUT_DIR}/aqi_spatial_distribution_interactive.html")

print("-" * 30)
print("Done!")